# Solutions — Decorators

Exercise notebook: `05_functions_and_oop/03_decorators.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Measure execution time

**Working solution**

In [ ]:
import functools
import time

def timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            wrapper.last_duration = time.perf_counter() - start
            wrapper.durations.append(wrapper.last_duration)
    wrapper.last_duration = None
    wrapper.durations = []
    return wrapper

**Explanation**

`try/finally` around the call records timing on both success and failure; `*args, **kwargs` forwards any signature.

**Complexity:** O(1) overhead.

**Common mistakes**

- Forgetting `@functools.wraps` → every decorated function is named 'wrapper' in logs.
- `time.time()` for durations — use the monotonic `perf_counter`.

## Exercise 2 — 🟡 Medium · Logging decorator factory

**Working solution**

In [ ]:
import functools
import logging

def log_calls(logger, level=logging.DEBUG):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            logger.log(level, "calling %s args=%r kwargs=%r", func.__name__, args, kwargs)
            try:
                result = func(*args, **kwargs)
            except Exception as exc:
                logger.error("%s raised %s: %s", func.__name__, type(exc).__name__, exc, exc_info=True)
                raise
            logger.log(level, "%s returned %r", func.__name__, result)
            return result
        return wrapper
    return decorator

**Explanation**

Three levels: `log_calls(logger)` returns `decorator`, which returns `wrapper`. Pass format args to the logger (`"%s", x`) instead of
f-strings so formatting is skipped when the level is disabled.

**Complexity:** O(1) overhead.

**Common mistakes**

- `@log_calls` without parentheses on a factory → the function itself is passed as `logger`.

## Exercise 3 — 🟡 Medium · Argument validation decorator

**Working solution**

In [ ]:
import functools
import inspect

def validate_args(**checks):
    def decorator(func):
        sig = inspect.signature(func)
        unknown = set(checks) - set(sig.parameters)
        if unknown:
            raise TypeError(f"{func.__name__} has no parameters {sorted(unknown)}")

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()
            for name, check in checks.items():
                value = bound.arguments[name]
                if not check(value):
                    raise ValueError(f"invalid value for {name}: {value!r}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

**Explanation**

`Signature.bind` maps positional and keyword arguments to parameter names exactly like the interpreter does, so validation works for every call style.

**Complexity:** O(params).

**Common mistakes**

- Only checking `kwargs` → positional calls bypass validation.
- Validating defaults is easy to forget — `apply_defaults()` handles it.

## Exercise 4 — 🟠 Hard · Retry with exponential backoff

**Working solution**

In [ ]:
import functools
import time

def retry(times=3, exceptions=(Exception,), delay=0.1, backoff=2.0, max_delay=30.0, sleep=time.sleep):
    if times < 1:
        raise ValueError("times must be >= 1")

    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            wait = delay
            for attempt in range(1, times + 1):
                wrapper.last_attempts = attempt
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if attempt == times:
                        raise
                    sleep(min(wait, max_delay))
                    wait *= backoff
        wrapper.last_attempts = 0
        return wrapper
    return decorator

**Explanation**

`except exceptions:` accepts a tuple of classes. `return` inside `try` exits on the first success; the bare `raise` on the last attempt preserves
the original traceback. Injecting `sleep` keeps tests instant. (Production code often adds jitter: `wait * random.uniform(0.5, 1.5)`.)

**Complexity:** O(times).

**Common mistakes**

- Retrying on *every* exception, including bugs like `TypeError`.
- Sleeping after the final attempt.
- Off-by-one: `times=3` meaning 3 *retries* (4 attempts).

## Challenge — 🔴 Very Hard · TTL + LRU cache decorator

**Working solution**

In [ ]:
import functools
import time
from collections import OrderedDict

def ttl_cache(ttl, maxsize=128, clock=time.monotonic):
    def decorator(func):
        entries = OrderedDict()          # key -> (expires_at, value)
        stats = {"hits": 0, "misses": 0}

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            key = (args, tuple(sorted(kwargs.items())))
            now = clock()
            hit = entries.get(key)
            if hit is not None and hit[0] > now:
                stats["hits"] += 1
                entries.move_to_end(key)
                return hit[1]
            stats["misses"] += 1
            value = func(*args, **kwargs)
            entries[key] = (now + ttl, value)
            entries.move_to_end(key)
            while len(entries) > maxsize:
                entries.popitem(last=False)
            return value

        def cache_info():
            return {**stats, "size": len(entries)}

        def cache_clear():
            entries.clear()
            stats.update(hits=0, misses=0)

        wrapper.cache_info = cache_info
        wrapper.cache_clear = cache_clear
        return wrapper
    return decorator

**Explanation**

Combines everything: decorator factory, closure state, `OrderedDict` LRU, injectable clock. Storing `(expires_at, value)` makes expiry a single comparison.

**Complexity:** O(1) per call.

**Common mistakes**

- Storing `created_at` and recomputing `now - created_at > ttl` everywhere (fine, but easy to get the boundary wrong).
- Never evicting → memory leak on high-cardinality keys.